# Load Path V2 System - Demonstration

This notebook demonstrates the new numba-compatible load path system.

**Key Features:**
- ✅ Numba-compatible calculations
- ✅ Follows variable_naming_protocol.ipynb
- ✅ Modular interface-based architecture
- ✅ Composable load paths
- ✅ Batch evaluation support
- ✅ Performance-focused design

**Focus:** Simple shear connections

**Load Path:** Beam web → Weld → Plate → Bolts → Column web

In [ ]:
import numpy as np
import time
from steel_lib.load_path_v2 import (
    SimpleShearConnection,
    evaluate_simple_shear_batch,
    find_optimal_design
)
from steel_lib.section_properties import create_aisc_section_selector
from steel_lib.plate_generator import generate_shear_plates
from steel_lib.weld_generator import generate_fillet_welds
from steel_lib.generator_combination import generate_bolt_configurations

print("Load Path V2 System - Ready!")
print("="*80)

## Example 1: Single Connection Evaluation

Evaluate a specific simple shear connection configuration.

In [ ]:
print("EXAMPLE 1: Single Connection Evaluation")
print("="*80)

# Load AISC database
aisc = create_aisc_section_selector('aisc-shapes-database-v16.0.xlsx')
print(f"✓ Loaded {aisc['database_info']['n_sections']} sections")

# Define beam section
beam_section = aisc['get_section_with_material']('W18X35', material='A992')
print(f"\nBeam: {beam_section['designation']} ({beam_section['material']})")
print(f"  d = {beam_section['d']:.2f} in, tw = {beam_section['tw']:.3f} in")
print(f"  F_y = {beam_section['F_y']:.1f} ksi, F_u = {beam_section['F_u']:.1f} ksi")

# Define column section
column_section = aisc['get_section_with_material']('W14X90', material='A992')
print(f"\nColumn: {column_section['designation']} ({column_section['material']})")
print(f"  d = {column_section['d']:.2f} in, tw = {column_section['tw']:.3f} in")
print(f"  F_y = {column_section['F_y']:.1f} ksi, F_u = {column_section['F_u']:.1f} ksi")

# Define plate configuration
plate_config = {
    'thickness': 0.375,  # 3/8" plate
    'width': 5.0,        # 5" wide
    'length': 12.0,      # 12" long
    'F_y': 50.0,         # A572 Grade 50
    'F_u': 65.0,
}
print(f"\nPlate: {plate_config['thickness']}" x {plate_config['width']}" x {plate_config['length']}"")
print(f"  F_y = {plate_config['F_y']:.1f} ksi, F_u = {plate_config['F_u']:.1f} ksi")

# Define weld configuration
weld_config = {
    'weld_size': 0.25,      # 1/4" fillet
    'weld_length': 12.0,    # 12" per side
    'electrode_id': 1,      # E70XX
    'both_sides': True,     # Both sides of plate
}
print(f"\nWeld: {weld_config['weld_size']}" fillet, E70XX")
print(f"  Length = {weld_config['weld_length']:.1f}" per side, both sides")

# Define bolt configuration
bolt_config = {
    'bolt_size': 0.75,      # 3/4" bolts
    'F_nv': 48.0,           # A325-N shear strength
    'F_nt': 90.0,           # A325-N tension strength
    'N_r': 4,               # 4 rows
    'N_c': 1,               # 1 column
    'S_r': 3.0,             # 3" row spacing
    'L_ev': 1.5,            # 1.5" edge distance
    'L_eh': 2.0,            # 2" edge distance
    'd_v': 0.8125,          # 13/16" hole
    'd_h': 0.8125,
}
print(f"\nBolts: {bolt_config['N_r']}x{bolt_config['N_c']} pattern, {bolt_config['bolt_size']}" A325-N")
print(f"  S_r = {bolt_config['S_r']:.1f}", L_ev = {bolt_config['L_ev']:.1f}"")

# Applied load
V_u = 40.0  # 40 kips shear
print(f"\nApplied Load: V_u = {V_u:.1f} kips")

# Create and evaluate connection
print("\n" + "-"*80)
print("Evaluating Connection...")
print("-"*80)

connection = SimpleShearConnection(
    beam_section=beam_section,
    plate_config=plate_config,
    weld_config=weld_config,
    bolt_config=bolt_config,
    column_section=column_section
)

result = connection.evaluate(V_u)

# Display results
print(f"\n{'RESULTS':^80}")
print("="*80)
print(f"  Connection is: {'✓ ADEQUATE' if result['is_adequate'] else '✗ INADEQUATE'}")
print(f"  Max Utilization: {result['max_utilization']:.1%}")
print(f"  Controlling: {result['controlling_interface']} - {result['controlling_limit_state']}")

print(f"\n  Interface Results:")
for i, interface_result in enumerate(result['interface_results']):
    status = "✓" if interface_result['is_adequate'] else "✗"
    print(f"    {i+1}. {interface_result['interface_type']:12} - {interface_result['limit_state']:20} "
          f"Util: {interface_result['utilization']:6.1%} {status}")

# Show detailed bolt interface results
bolt_result = result['interface_results'][1]
if 'all_capacities' in bolt_result:
    print(f"\n  Bolt Interface - All Limit States:")
    for state, cap in bolt_result['all_capacities'].items():
        util = bolt_result['all_utilizations'][state]
        print(f"    {state:20} Capacity: {cap:6.1f} kips, Util: {util:6.1%}")

print("="*80)

## Example 2: Batch Evaluation - Multiple Configurations

Evaluate multiple connection configurations to find optimal design.

In [ ]:
print("EXAMPLE 2: Batch Evaluation")
print("="*80)
print("Evaluating multiple connection configurations...")

# Same beam and column
beam_section = aisc['get_section_with_material']('W18X35', material='A992')
column_section = aisc['get_section_with_material']('W14X90', material='A992')

print(f"\nBeam: {beam_section['designation']}")
print(f"Column: {column_section['designation']}")
print(f"Applied Load: V_u = {V_u:.1f} kips")

# Generate multiple configurations
print("\n" + "-"*80)
print("Generating Configurations...")
print("-"*80)

plates = generate_shear_plates(
    plate_grade_id=[1],                 # A572_50
    thickness=[0.375, 0.5],             # 3/8", 1/2"
    width=[5.0],                        # 5"
    length=[12.0, 15.0]                 # 12", 15"
)

welds = generate_fillet_welds(
    electrode_id=[1],                   # E70XX
    weld_size=[0.1875, 0.25],          # 3/16", 1/4"
    weld_length=[12.0, 15.0],
    both_sides=[True]
)

bolts = generate_bolt_configurations(
    bolt_size=np.array([0.75], dtype=np.float64),
    bolt_grade_id=np.array([0], dtype=np.int64),
    member_a_BHT_id=np.array([0], dtype=np.int64),
    member_b_BHT_id=np.array([0], dtype=np.int64),
    N_r=np.array([4, 5], dtype=np.int64),
    S_r=np.array([3.0], dtype=np.float64),
    N_c=np.array([1], dtype=np.int64),
    S_c=np.array([3.0], dtype=np.float64),
    L_ev=np.array([1.5], dtype=np.float64),
    L_eh=np.array([2.0], dtype=np.float64),
    Ga=np.array([3.0], dtype=np.float64)
)

print(f"✓ Generated {len(plates['thickness'])} plate configs")
print(f"✓ Generated {len(welds['weld_size'])} weld configs")
print(f"✓ Generated {len(bolts['bolt_size'])} bolt configs")

total_configs = len(plates['thickness']) * len(welds['weld_size']) * len(bolts['bolt_size'])
print(f"\nTotal configurations: {total_configs}")

# Batch evaluation
print("\n" + "-"*80)
print("Evaluating All Configurations...")
print("-"*80)

start = time.time()
batch_results = evaluate_simple_shear_batch(
    beam_sections=beam_section,
    plate_configs=plates,
    weld_configs=welds,
    bolt_configs=bolts,
    column_section=column_section,
    V_u=V_u
)
eval_time = (time.time() - start) * 1000

print(f"\n✓ Evaluated {batch_results['count']} configurations in {eval_time:.2f} ms")
print(f"  Time per configuration: {eval_time/batch_results['count']:.3f} ms")

# Analyze results
n_adequate = np.sum(batch_results['results_adequate'])
print(f"\n  Adequate designs: {n_adequate} / {batch_results['count']} ({n_adequate/batch_results['count']:.1%})")

if n_adequate > 0:
    adequate_utils = batch_results['results_utilization'][batch_results['results_adequate']]
    print(f"  Utilization range: {adequate_utils.min():.1%} to {adequate_utils.max():.1%}")
    
    # Find optimal
    optimal = find_optimal_design(batch_results)
    
    if optimal['found']:
        print(f"\n  OPTIMAL DESIGN:")
        print(f"    Index: {optimal['index']}")
        print(f"    Utilization: {optimal['utilization']:.1%}")
        print(f"    Controlling: {optimal['controlling_state']}")
else:
    print(f"  ⚠ No adequate designs found!")
    print(f"  Consider: Larger plates, more bolts, or bigger welds")

print("="*80)

## Example 3: Performance Comparison

Compare evaluation speed for different problem sizes.

In [ ]:
print("EXAMPLE 3: Performance Benchmarking")
print("="*80)

# Test different problem sizes
test_cases = [
    {'plates': 2, 'welds': 2, 'bolts': 2, 'name': 'Small'},
    {'plates': 4, 'welds': 4, 'bolts': 4, 'name': 'Medium'},
    {'plates': 8, 'welds': 8, 'bolts': 4, 'name': 'Large'},
]

print(f"\n{'Size':<10} {'Configs':<10} {'Time (ms)':<12} {'Per Config (ms)':<15}")
print("-"*50)

for test in test_cases:
    # Generate configs
    plates = generate_shear_plates(
        plate_grade_id=[1],
        thickness=np.linspace(0.25, 0.75, test['plates']),
        width=[5.0],
        length=[12.0]
    )
    
    welds = generate_fillet_welds(
        electrode_id=[1],
        weld_size=np.linspace(0.1875, 0.375, test['welds']),
        weld_length=[12.0],
        both_sides=[True]
    )
    
    bolts = generate_bolt_configurations(
        bolt_size=np.array([0.75], dtype=np.float64),
        bolt_grade_id=np.array([0], dtype=np.int64),
        member_a_BHT_id=np.array([0], dtype=np.int64),
        member_b_BHT_id=np.array([0], dtype=np.int64),
        N_r=np.arange(3, 3+test['bolts'], dtype=np.int64),
        S_r=np.array([3.0], dtype=np.float64),
        N_c=np.array([1], dtype=np.int64),
        S_c=np.array([3.0], dtype=np.float64),
        L_ev=np.array([1.5], dtype=np.float64),
        L_eh=np.array([2.0], dtype=np.float64),
        Ga=np.array([3.0], dtype=np.float64)
    )
    
    n_configs = len(plates['thickness']) * len(welds['weld_size']) * len(bolts['bolt_size'])
    
    # Evaluate
    start = time.time()
    batch_results = evaluate_simple_shear_batch(
        beam_sections=beam_section,
        plate_configs=plates,
        weld_configs=welds,
        bolt_configs=bolts,
        column_section=column_section,
        V_u=40.0
    )
    eval_time = (time.time() - start) * 1000
    
    print(f"{test['name']:<10} {n_configs:<10} {eval_time:<12.2f} {eval_time/n_configs:<15.3f}")

print("="*80)
print("\n✓ Load Path V2 System demonstrates excellent performance!")
print("  - Numba-compatible calculations")
print("  - Follows AISC variable naming")
print("  - Modular and maintainable architecture")
print("  - Ready for integration with aisc_14th module")

## Next Steps

**To integrate with aisc_14th module:**

1. Replace placeholder calculations with actual AISC limit state functions:
   - `transfer_weld_to_plate()` → Use weld calculations from aisc_14th
   - `transfer_bolts_shear()` → Use `bolt_shear()` from aisc_14th
   - `check_bolt_bearing()` → Use `bolt_bearing()` from aisc_14th
   - `check_plate_shear_yielding()` → Use `shear_yielding_rupture()` from aisc_14th
   - `check_block_shear()` → Use `block_shear()` from aisc_14th

2. Add more interface types:
   - Moment connections
   - Braced connections
   - Column base plates

3. Enhance batch evaluation:
   - Multiple beam-column combinations
   - Multiple load cases
   - Optimization algorithms

**Key advantages of this architecture:**
- Each interface is independent and testable
- Easy to add new limit states
- Composable for different connection types
- Performance-optimized with numba
- Clear separation of concerns